In [1]:
import ast

import pandas as pd
from tqdm.notebook import tqdm

tqdm.pandas()

In [2]:
%%time
df = pd.read_csv("notes_with_extracted_urls.csv")

<timed exec>:1: DtypeWarning: Columns (5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.


CPU times: user 14.5 s, sys: 1.31 s, total: 15.8 s
Wall time: 15.8 s


In [3]:
rows, cols = df.shape
print(f"{rows:,} rows × {cols:,} columns")

1,855,148 rows × 24 columns


In [4]:
df.head()

,noteId,noteAuthorParticipantId,createdAtMillis,tweetId,classification,believable,harmful,validationDifficulty,misleadingOther,misleadingFactualError,...,misleadingSatire,notMisleadingOther,notMisleadingFactuallyCorrect,notMisleadingOutdatedButNotWhenWritten,notMisleadingClearlySatire,notMisleadingPersonalOpinion,trustworthySources,summary,isMediaNote,summary_urls
0,1783179305159200982,C784F04F26E124F4D6EC01658D8F5565005D3092741FB3...,1713978050878,1783159712986382830,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,1,The House failed to pass a border protection l...,0,['https://sourcenm.com/2024/04/22/u-s-house-vo...
1,1783181538789605871,C784F04F26E124F4D6EC01658D8F5565005D3092741FB3...,1713978583415,1783171851818021181,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,0,1,...,0,0,0,0,0,0,1,The United States has 50 States https://da...,0,['https://data.census.gov/all/profiles?q=All%2...
2,1783182562279494134,C784F04F26E124F4D6EC01658D8F5565005D3092741FB3...,1713978827435,1783154445682979015,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,1,TikTok only mentions “ban” and chooses to igno...,0,['https://www.reuters.com/world/us/senators-ho...
3,1883711635770196070,C784F04F26E124F4D6EC01658D8F5565005D3092741FB3...,1737946826294,1883619411774345444,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,1,0,...,0,0,0,0,0,0,1,This could be considered a threat https://...,0,['https://www.aol.com/finance/d-c-federal-cour...
4,1537142913737428992,5684B38EB58FD8BE75ABA37F0BE040EC70380B002ADF9D...,1655318404027,1377030478167937024,MISINFORMED_OR_POTENTIALLY_MISLEADING,BELIEVABLE_BY_MANY,CONSIDERABLE_HARM,EASY,0,1,...,0,0,0,0,0,0,1,Forbes has a good rundown of the investigation...,0,['https://www.forbes.com/sites/rachelsandler/2...


In [5]:
df["summary_urls"].dtype

dtype('O')

In [6]:
len(ast.literal_eval(df["summary_urls"].iloc[14]))

2

In [7]:
df["num_urls"] = df["summary_urls"].progress_apply(
    lambda x: len(ast.literal_eval(x)) if pd.notnull(x) else 0
)

  0%|          | 0/1855148 [00:00<?, ?it/s]

In [9]:
print(sorted(df["num_urls"].unique().tolist()))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 44, 46, 47, 49, 134]


In [21]:
# row_with_12_urls = df[df['num_urls'] == 134]
# row_with_12_urls[['noteId','summary', 'summary_urls']].head(5).to_csv('temp.csv', index=False)

In [13]:
def find_invalid_urls(url_string):
    invalids = []
    if pd.isnull(url_string):
        return invalids
    try:
        urls = ast.literal_eval(url_string)
        for url in urls:
            if not (url.startswith("http") or url.startswith("www")):
                invalids.append(url)
    except (ValueError, SyntaxError):
        # If the string can't be parsed, treat it as invalid
        invalids.append(url_string)
    return invalids

In [14]:
%%time
df["invalid_urls"] = df["summary_urls"].progress_apply(find_invalid_urls)

  0%|          | 0/1855148 [00:00<?, ?it/s]

CPU times: user 27.1 s, sys: 179 ms, total: 27.3 s
Wall time: 27.2 s


In [15]:
all_invalid_urls = [url for sublist in df["invalid_urls"] for url in sublist]

In [16]:
len(all_invalid_urls)

29366

In [17]:
cnt = 0
print("All invalid URLs:")
for url in all_invalid_urls:
    print(url)

    if cnt == 30:
        break
    cnt = cnt + 1

All invalid URLs:
usatoday.com/story/news/politics/2024/12/23/what-is-commuting-sentence-death-row-joe-biden/77172737007/
congress.gov/bill/119th-congress/house-bill/191/cosponsors
fred.stlouisfed.org/series/SP500
cedars-sinai.org/newsroom/new-study-is-there-a-link-between-covid-19-vaccination-and-pots/
Dating.com
US.https://en.wikipedia.org/wiki/List_of_United_States_cities_by_crime_rate
help.twitter.com/en/rules-and-p…
conservative.ca
RedState.com
bit.ly
Lego.com
Bild.de
eadaily.com
avio.pro
eadaily.com
avio.pro
ED.Gov
creativebloq.com
X.com
x.com
x.com
twitter.com
preciosclaros.gob.ar
nic.ar
reuters.com/article/uk-factcheck-altered-photo-ilhan-omar/fact-check-altered-photograph-of-congresswoman-ilhan-omar-idUSKBN28S2SA
finance.google.com
Temu.com
arketf.net
elDiario.es
superdutydayusl.shop
neptune.ai


In [19]:
# with open("invalid_urls.txt", "w") as f:
#     for url in all_invalid_urls:
#         f.write(url + "\n")

In [33]:
num_zero_url_rows = (df["num_urls"] == 0).sum()
print(f"Number of rows with 0 URLs: {num_zero_url_rows}")

Number of rows with 0 URLs: 327410


In [34]:
non_zero_percent = (df["num_urls"] != 0).mean() * 100
print(f"Percentage of rows with non-zero URLs: {non_zero_percent:.2f}%")

Percentage of rows with non-zero URLs: 82.35%
